# Deret Waktu Astrofisika - Analisis Siklus 

## 1. Konteks Fisis
Bintik Matahari (Sunspots) adalah fenomena sementara pada fotosfer Matahari yang tampak sebagai area gelap dibandingkan daerah sekitarnya. Bintik-bintik ini merupakan wilayah dengan suhu permukaan yang lebih rendah akibat konsentrasi fluks medan magnet yang menghambat proses konveksi.

Pengamatan historis terhadap bintik matahari telah dilakukan selama berabad-abad dan menunjukkan adanya periodisitas yang jelas dalam aktivitas magnetik Matahari. Variabilitas ini memiliki dampak penting terhadap cuaca antariksa (space weather), komunikasi satelit, serta kondisi atmosfer bagian atas Bumi.

Data jumlah bintik matahari bulanan dianalisis menggunakan metode Fast Fourier Transform (FFT), yaitu implementasi efisien dari Discrete Fourier Transform (DFT), untuk mengubah data dari domain waktu ke domain frekuensi.

Melalui analisis spektrum frekuensi, diperoleh frekuensi dominan yang kemudian dikonversi menjadi periode siklus aktivitas Matahari. Periode dominan yang teridentifikasi dikenal sebagai Schwabe Cycle, yaitu siklus aktivitas magnetik Matahari dengan periode sekitar 11 tahun.

## 2. Nilai Harapan

# Analisis FFT untuk Menentukan Siklus Dominan Sunspot

Dataset menyediakan jumlah sunspot $y_n$ pada interval waktu diskret yang berjarak sama. Interval pengambilan sampel adalah **satu bulan**, sehingga:

$$
\Delta t = \frac{1}{12}\ \text{tahun}
$$

Anda akan menghitung **Fast Fourier Transform (FFT)** untuk memperoleh komponen pada domain frekuensi, yaitu $Y_k$.

Untuk menemukan siklus fisik yang dominan, lakukan langkah-langkah berikut:

## Langkah-langkah

1. Hitung **spektrum daya ter-normalisasi**:

   $$
   P_k = \frac{2}{N}|Y_k|
   $$

2. Bentuk array frekuensi $f_k$ yang sesuai dengan setiap bin frekuensi hasil FFT.

3. Identifikasi frekuensi $f_{\max}$ yang menghasilkan nilai daya spektral maksimum.

4. Konversikan frekuensi tersebut menjadi periode waktu fisik $T$ (dalam tahun) menggunakan hubungan:

   $$
   T = \frac{1}{f}
   $$

## Tujuan

Tentukan periode dominan $T$ yang merepresentasikan siklus utama aktivitas sunspot berdasarkan hasil analisis FFT.

In [ ]:
# Memasukkan librari untuk komputasi saintifik (Numpy, plot, dan Panda untuk baca file .csv)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import pandas as pd

# mengatur style plot sesuai dengan template
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 3. Pemuatan Data dan Visualisasi Domain Waktu

Dataset `sunspot_data.csv` yang berada pada direktori `data/` dimuat. Dataset ini terdiri atas dua kolom, yaitu `Fractional_Year` (contohnya 1850.042) dan `Sunspot_Count`. Data mentah kemudian diplot pada domain waktu sehingga pola periodisitas serta keberadaan derau (*noise*) dapat diidentifikasi melalui pengamatan visual.

In [ ]:
# --- TASK 1: Memasukkan dan Mem-plot data dalam domain frekuensi-waktu---
!wget -O sunspots.csv https://raw.githubusercontent.com/TR2Theo/fft-case-studies-template/main/data/sunspots.csv # Pastikan CSV mentah diunduh, karena wget sebelumnya mungkin telah mengunduh HTML
df = pd.read_csv("sunspots.csv") # 1. Memasukkan data sunspot.csv dengan panda
time = pd.to_datetime(df["Date"]) # 2. Ekstrak array waktu (t) dan 
y = df["Monthly Mean Total Sunspot Number"].values # 2. array sinyal (y)
fig, ax = plt.subplots()

# 3. Plot Sunspot Count vs. Time
ax.plot(time, y)

# 4. Judul dan label
ax.set_title("Sunspot Data - Domain Waktu")
ax.set_xlabel("Time (Tahun)")
ax.set_ylabel("Jumlah Titik Sunspot")

# Mengatur label tahun agar sesuai posisi datetime
ax.xaxis.set_major_locator(mdates.YearLocator(10))
# label tiap 10 tahun

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Rotasi label tahun
plt.xticks(rotation=45)

# Grid dan layout
plt.grid(True)
plt.tight_layout()
plt.savefig('Sunspot_Data-Domain_Waktu.png', dpi=300)
plt.show()

## 4. Penghapusan Komponen DC

Dataset fisika yang hanya mengandung nilai positif (seperti jumlah bintik matahari) umumnya memiliki nilai rata-rata (mean) yang tidak nol. Dalam domain frekuensi, kondisi ini menghasilkan puncak yang sangat besar pada frekuensi $f = 0$ Hz yang dikenal sebagai komponen DC (Direct Current Component). Puncak tersebut dapat mendominasi spektrum frekuensi sehingga osilasi periodik yang menjadi fokus analisis menjadi sulit diamati.

Nilai rata-rata (mean) dari data jumlah bintik matahari dihitung, kemudian dikurangkan dari setiap data pada sinyal asli sehingga diperoleh sinyal yang telah dipusatkan terhadap nol (mean-centered signal). Selanjutnya, Transformasi Fourier Cepat (FFT) diterapkan pada sinyal hasil pemusatan tersebut.

In [ ]:
# --- TUGAS 2: Lepas komponen DC(direct current) ---
Mean = np.mean(y) # 1. Hitung rata-rata susunan bintik matahari.
y_centered = y - Mean   # 2. Buat array baru untuk y yang dipusatkan
# 3. Plot data terpusat untuk memverifikasi bahwa data tersebut berosilasi di sekitar nol.
fig, ax = plt.subplots()

#3. plot data centered 
ax.plot(time, y_centered)

# Judul dan label
ax.set_title("Sunspot Data - Centered")
ax.set_xlabel("Time (Tahun)")
ax.set_ylabel("Jumlah Titik Sunspot")
# Mengatur label tahun agar sesuai posisi datetime
ax.xaxis.set_major_locator(mdates.YearLocator(10))
# label tiap 10 tahun

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Rotasi label tahun
plt.xticks(rotation=45)

# Grid dan layout
plt.grid(True)
plt.tight_layout()
plt.savefig('Sunspot_Data-Centered.png', dpi=300)
plt.show()


## 5. Transformasi Data ke Domain Frekuensi

Algoritma Fast Fourier Transform (FFT) diimplementasikan pada data yang telah dipusatkan (*centered data*). Selanjutnya, sumbu frekuensi yang bersesuaian juga dibangun. Perhatian khusus diberikan pada kesesuaian satuan yang digunakan. Apabila interval sampling ($\Delta t$) dinyatakan dalam tahun, maka frekuensi yang dihasilkan akan memiliki satuan siklus per tahun ($yr^{-1}$).

In [ ]:
# --- TASK 3: Hitung FFT dan Spektrum Daya ---
N = len(y) # 1. Define N (number of samples) and
dt = 1/12  #dt (sampling interval in years).
fft_result = np.fft.fft(y_centered) # 2. menghitung FFT menggunakan np.fft.fft()
freq_bins = np.fft.fftfreq(N, d=dt) # 3. menghitung kelompok rentang frekuensi tertentu yang digunakan untuk memecah sinyal kompleks menggunakan np.fft.fftfreq()

# 4. Hitung spektrum magnitudo positif (Daya). Ingatlah untuk hanya menjaga separuh frekuensi positif (hingga batas Nyquist).
positive_mask = freq_bins >= 0
positive_freq = freq_bins[positive_mask]
magnitude = np.abs(fft_result[positive_mask])
# ------------------------------------------------
# Normalisasi Power spektrum
# ------------------------------------------------
power = (2 / N)*magnitude

# penentuan filter frekuensi rendah untuk menentukan batas dalam plt.xlim()
n_classes = 20 #jumlah kelas tiap data

# frekuensi tiap interval
bins = np.linspace(
    positive_freq.min(),
    positive_freq.max(),
    n_classes + 1
)
# menyimpan total power tiap kelas
class_power = []

# =========================================================
# menghitung total power tiap kelas
# =========================================================

for i in range(n_classes):

    mask = (
        (positive_freq >= bins[i]) &
        (positive_freq < bins[i+1])
    )

    total_power = np.sum(power[mask])

    class_power.append(total_power)

class_power = np.array(class_power)

# =========================================================
# Mencari kelas yang power spetrumnya dominan
# =========================================================
dominant_idx = np.argmax(class_power)
upper_bound = bins[dominant_idx + 1] #batas atas kelas yang nilai powernya dominan


## 6. Power Spectrum Analysis

**Task 4:** Plot the power spectrum (Magnitude vs. Frequency). Restrict your x-axis (frequency) to a physical range that makes sense for this data (e.g., $0$ to $0.5$ cycles/year) to zoom in on the relevant peaks.

In [ ]:
# --- TASK 4: Visualisasi domain frekuensi ---
plt.plot(positive_freq, power) # 1. Plot the power spectrum.
plt.title("Sunspot Data - Frequency Domain")
# 2. menamakan label x-axis sebagai "Frequency (cycles/year)" dan y-axis sebagai "Magnitude".
plt.xlabel("Frequency (cycles/year)")
plt.ylabel("Magnitude")
plt.xlim(0, upper_bound) # 3. menggunakan plt.xlim() untuk memfilter di frekuensi rendah
plt.grid(True)
plt.savefig('power_spektrum.png', dpi=300)
plt.show()

## 7. Conclusions and Physical Interpretation

**Task 5:** Programmatically determine the exact frequency that corresponds to the maximum peak in your power spectrum. Convert this frequency into a time period (years). 

Provide a brief written analysis of your findings below. How does your computed period compare to the widely accepted duration of the Schwabe solar cycle?

In [ ]:
# --- TASK 5: Program untuk mendeteksi puncak frekuensi ---
# mengabaikan frekuensi 0 agar tidak mendektesi komponen DC
peak_index = np.argmax(power[1:]) + 1
# Frekuensi dominan
dominant_frequency = positive_freq[peak_index]
# Menghitung periode dominan
dominant_period = 1 / dominant_frequency
# Plot frekuensi-time domain dengan menandai titik power spektrum tertinggi
plt.figure(figsize=(12,5))

plt.plot(positive_freq, power, label='Power Spectrum')

plt.scatter(
    dominant_frequency,
    power[peak_index],
    s=100,
    label='Dominant Peak'
)

plt.xlim(0, upper_bound)

plt.title("Dominant Frequency Peak")
plt.xlabel("Frequency (cycles/year)")
plt.ylabel("Magnitude")

plt.legend()
plt.grid(True)
plt.savefig('power_spektrum_peak.png', dpi=300)
plt.show()

#urutkan power untuk mencari first peak dan secondary peak
# Menunjukkan hasil dalam bentuk string
peak_ind = []
powersorted = sorted(power, reverse=True)
t = 1
for i in powersorted[:3]:
  print("=================================================")
  print(f"POWER SPECTRUM RESULT {t}")
  print("=================================================")
  peak = np.where(power == i)[0][0]
  domfrequency = positive_freq[peak]
  domperiod = 1/domfrequency
  print("FFT PEAK DETECTION RESULT")
  print("=================================================")

  print(f"Dominant Frequency            : {domfrequency:.4f} cycles/year")
  print(f"Computed Solar Cycle Period   : {domperiod:.2f} years")

  print("=================================================")
  t += 1

### Final Analysis
*(Double click this cell to edit)*

**Computed Solar Cycle Period:** [Insert your numerical result here] years.

**Physical Interpretation:**
[Write a 3-4 sentence paragraph discussing the success of the FFT in isolating the physical cycle despite the high level of noise in the historical data. Discuss any secondary peaks you might observe in the spectrum, and what they might represent physically or mathematically.]